# Models 

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from itertools import product

In [2]:
# import data
train = pd.read_parquet('../data/model/final_train.parquet')
val = pd.read_parquet('../data/model/final_val.parquet')
test = pd.read_parquet('../data/model/final_test.parquet')

In [3]:
# check the min and max Date in each dataset
print("Train Date range:", train['Date'].min(), "to", train['Date'].max())
print("Validation Date range:", val['Date'].min(), "to", val['Date'].max())
print("Test Date range:", test['Date'].min(), "to", test['Date'].max())

Train Date range: 2016-01-04 00:00:00 to 2021-12-30 00:00:00
Validation Date range: 2022-01-03 00:00:00 to 2022-12-30 00:00:00
Test Date range: 2023-01-03 00:00:00 to 2023-12-29 00:00:00


In [4]:
X_train = train.loc[:, train.columns != "return_next_day"]
X_val = val.loc[:, val.columns != "return_next_day"]
X_test = test.loc[:, test.columns != "return_next_day"]
X_full = pd.concat([X_train, X_val, X_test], axis=0).reset_index(drop=True)

In [6]:
X_train.columns

Index(['Date', 'tic', 'Open', 'High', 'Low', 'Close', 'Volume',
       'sales_growth_qoq', 'sales_growth_ttm', 'asset_growth',
       ...
       'pca_emb_54', 'pca_emb_55', 'pca_emb_56', 'pca_emb_57', 'pca_emb_58',
       'pca_emb_59', 'pca_emb_60', 'pca_emb_61', 'pca_emb_62', 'pca_emb_63'],
      dtype='object', length=108)

In [5]:
y_train = train[['Date', 'tic', 'return_next_day']]
y_val = val[['Date', 'tic', 'return_next_day']]
y_test = test[['Date', 'tic', 'return_next_day']]
y_full = pd.concat([y_train, y_val, y_test], axis=0).reset_index(drop=True)

In [22]:
def directional_accuracy(y_true, y_pred):
    return (np.sign(y_true) == np.sign(y_pred)).mean()

#### Model 1 - using features from baseline 

From the baseline, we found a couple of features that are helpful to predict next day return 
- s4_dow_mean
- s7_roll63_mean_return
- ema_ema_50, ema_ema_20, ema_ema_10
- b0 (simple lag return)
- ma_1m, ma_3d, ma_14d, ma_30d, ma_50d, ma_2w, ma_2d, ma_1w
- s3_month_of_year_mean

In [7]:
def add_rolling_mean_feature(df, group_col, value_col, window, feature_name, shift_before=True):
    """
    Rolling mean feature per group_col.

    If shift_before=True, shift by 1 so feature at time t only uses info before t.
    """
    df = df.copy()
    df[feature_name] = (
        df.groupby(group_col)[value_col]
          .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    )
    if shift_before:
        df[feature_name] = df.groupby(group_col)[feature_name].shift(1)
    return df


def add_ema_feature(df, group_col, value_col, span, feature_name, shift_before=True):
    """
    Exponential moving average feature per group_col.

    If shift_before=True, shift by 1 so there is no look-ahead.
    """
    df = df.copy()
    df[feature_name] = (
        df.groupby(group_col)[value_col]
          .transform(lambda x: x.ewm(span=span, adjust=False, min_periods=1).mean())
    )
    if shift_before:
        df[feature_name] = df.groupby(group_col)[feature_name].shift(1)
    return df

In [15]:
def compute_signal_features_from_Xfull(X_full):
    """
    X_full must have at least: Date, tic, Close.
    We compute ret internally and build all the signal features.

    Returns a DataFrame with:
        ['Date', 'tic',
         's4_dow_mean',
         's7_roll63_mean_return',
         'ema_ema_50', 'ema_ema_20', 'ema_ema_10',
         'b0',
         'ma_1m', 'ma_3d', 'ma_14d', 'ma_30d', 'ma_50d',
         'ma_2w', 'ma_2d', 'ma_1w',
         's3_month_of_year_mean']
    """
    df = X_full[['Date', 'tic', 'Close']].copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['tic', 'Date']).reset_index(drop=True)

    # --- realized daily return (today vs yesterday) ---
    df['ret'] = df.groupby('tic')['Close'].pct_change()

    # --- b0: previous day's return per ticker ---
    df['b0'] = df.groupby('tic')['ret'].shift(1)

    # --- s4_dow_mean: prior day's cross-sectional mean ret (market-like) ---
    # cross-sectional mean ret per day
    df['dow_mean_ret'] = df.groupby('Date')['ret'].transform('mean')
    # use previous day's dow_mean_ret for each ticker to avoid look-ahead
    df['s4_dow_mean'] = df.groupby('tic')['dow_mean_ret'].shift(1)

    # --- s7_roll63_mean_return: 63-day rolling mean of ret per ticker ---
    df = add_rolling_mean_feature(
        df,
        group_col='tic',
        value_col='ret',
        window=63,
        feature_name='s7_roll63_mean_return',
        shift_before=True,
    )

    # --- EMA features on ret per ticker ---
    df = add_ema_feature(
        df,
        group_col='tic',
        value_col='ret',
        span=50,
        feature_name='ema_ema_50',
        shift_before=True,
    )
    df = add_ema_feature(
        df,
        group_col='tic',
        value_col='ret',
        span=20,
        feature_name='ema_ema_20',
        shift_before=True,
    )
    df = add_ema_feature(
        df,
        group_col='tic',
        value_col='ret',
        span=10,
        feature_name='ema_ema_10',
        shift_before=True,
    )

    # --- Moving-average features on ret per ticker ---
    ma_windows = {
        'ma_2d': 2,
        'ma_3d': 3,
        'ma_14d': 14,
        'ma_1w': 5,   # ~1 week
        'ma_2w': 10,  # ~2 weeks
        'ma_1m': 21,  # ~1 month
        'ma_30d': 30,
        'ma_50d': 50,
    }

    for label, w in ma_windows.items():
        df = add_rolling_mean_feature(
            df,
            group_col='tic',
            value_col='ret',
            window=w,
            feature_name=label,
            shift_before=True,
        )

    # --- s3_month_of_year_mean: expanding month-of-year mean ret per (tic, month) ---
    df['month'] = df['Date'].dt.month
    df = df.sort_values(['tic', 'month', 'Date']).reset_index(drop=True)

    def _expanding_month_mean(s):
        # expanding mean of past values only
        return s.shift(1).expanding(min_periods=1).mean()

    df['s3_month_of_year_mean'] = (
        df.groupby(['tic', 'month'])['ret'].transform(_expanding_month_mean)
    )

    # Sort back by tic, Date
    df = df.sort_values(['tic', 'Date']).reset_index(drop=True)

    feature_cols = [
        's4_dow_mean',
        's7_roll63_mean_return',
        'ema_ema_50',
        'ema_ema_20',
        'ema_ema_10',
        'b0',
        'ma_1m',
        'ma_3d',
        'ma_14d',
        'ma_30d',
        'ma_50d',
        'ma_2w',
        'ma_2d',
        'ma_1w',
        's3_month_of_year_mean',
    ]

    # Fill NaNs in features with 0 (no historical info yet → neutral signal)
    df[feature_cols] = df[feature_cols].fillna(0.0)

    features_df = df[['Date', 'tic'] + feature_cols].copy()
    return features_df

In [16]:
features_full = compute_signal_features_from_Xfull(X_full)
features_full.head()

,Date,tic,s4_dow_mean,s7_roll63_mean_return,ema_ema_50,ema_ema_20,ema_ema_10,b0,ma_1m,ma_3d,ma_14d,ma_30d,ma_50d,ma_2w,ma_2d,ma_1w,s3_month_of_year_mean
0,2016-01-04,A,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2016-01-05,A,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2016-01-06,A,0.002172,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440
3,2016-01-07,A,-0.015380,0.000499,-0.003131,-0.002690,-0.002008,0.004439,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499
4,2016-01-08,A,-0.024183,-0.013825,-0.004674,-0.006479,-0.009365,-0.042474,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.019018,-0.013825,-0.013825


In [17]:
# Augment design matrices with signal features
X_train_aug = X_train.merge(features_full, on=['Date', 'tic'], how='left')
X_val_aug   = X_val.merge(features_full,   on=['Date', 'tic'], how='left')
X_test_aug  = X_test.merge(features_full,  on=['Date', 'tic'], how='left')

# Optional sanity checks
print("X_train:", X_train.shape, "->", X_train_aug.shape)
print("X_val:  ", X_val.shape,   "->", X_val_aug.shape)
print("X_test: ", X_test.shape,  "->", X_test_aug.shape)

# Check for missing values in new features (should be minor if at all)
X_train_aug.isna().mean().sort_values().tail(15)

X_train: (729659, 108) -> (729659, 123)
X_val:   (124747, 108) -> (124747, 123)
X_test:  (124727, 108) -> (124727, 123)


pca_emb_0                0.0
news_count               0.0
sum_sentiment            0.0
min_sentiment            0.0
max_sentiment            0.0
mean_sentiment           0.0
yield_spread_10y_2y      0.0
sp500                    0.0
vix                      0.0
aaa_yield                0.0
t3m                      0.0
t2y                      0.0
t10y                     0.0
retail_sales             0.0
s3_month_of_year_mean    0.0
dtype: float64

In [18]:
feature_cols = [
    's4_dow_mean',
    's7_roll63_mean_return',
    'ema_ema_50',
    'ema_ema_20',
    'ema_ema_10',
    'b0',
    'ma_1m',
    'ma_3d',
    'ma_14d',
    'ma_30d',
    'ma_50d',
    'ma_2w',
    'ma_2d',
    'ma_1w',
    's3_month_of_year_mean',
]

In [19]:
# columns we want to keep
keep_cols = ['Date', 'tic', 'Close'] + feature_cols

X_train_sig = X_train_aug[keep_cols].copy()
X_val_sig   = X_val_aug[keep_cols].copy()
X_test_sig  = X_test_aug[keep_cols].copy()

In [20]:
X_train_sig.head()

,Date,tic,Close,s4_dow_mean,s7_roll63_mean_return,ema_ema_50,ema_ema_20,ema_ema_10,b0,ma_1m,ma_3d,ma_14d,ma_30d,ma_50d,ma_2w,ma_2d,ma_1w,s3_month_of_year_mean
0,2016-01-04,A,37.636356,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2016-01-05,A,37.506878,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2016-01-06,A,37.673370,0.002172,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440
3,2016-01-07,A,36.073215,-0.015380,0.000499,-0.003131,-0.002690,-0.002008,0.004439,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499
4,2016-01-08,A,35.693970,-0.024183,-0.013825,-0.004674,-0.006479,-0.009365,-0.042474,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.019018,-0.013825,-0.013825


In [21]:
# Features
X_train_m = X_train_sig.drop(columns=["Date", "tic"])
X_val_m   = X_val_sig.drop(columns=["Date", "tic"])
X_test_m  = X_test_sig.drop(columns=["Date", "tic"])

# Targets
y_train_vec = y_train["return_next_day"]
y_val_vec   = y_val["return_next_day"]
y_test_vec  = y_test["return_next_day"]

In [29]:
lr = LinearRegression()
lr.fit(X_train_m, y_train_vec)

# predictions
pred_val_lr = lr.predict(X_val_m)
pred_test_lr = lr.predict(X_test_m)

# evaluation
print("Linear Regression:")
print("Val DA:", directional_accuracy(y_val_vec, pred_val_lr))

print("Test DA:", directional_accuracy(y_test_vec, pred_test_lr))

Linear Regression:
Val DA: 0.4866890586547171
Test DA: 0.4956905882447265


In [43]:
# optional but helpful: cast to float32
X_train_m_32 = X_train_m.astype("float32")
X_val_m_32   = X_val_m.astype("float32")
X_test_m_32  = X_test_m.astype("float32")

y_train_vec = y_train_vec.astype("float32")
y_val_vec   = y_val_vec.astype("float32")
y_test_vec  = y_test_vec.astype("float32")

xgb_model = xgb.XGBRegressor(
    n_estimators=2000,      # large cap; early stopping will cut it
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",     # fast CPU algorithm
    n_jobs=-1,              # use all CPU cores
    random_state=42,
    eval_metric="rmse",     # <-- move eval_metric here
)

xgb_model.fit(
    X_train_m_32,
    y_train_vec,
    eval_set=[(X_val_m_32, y_val_vec)],
    verbose=False,
)

# Use the best iteration found by early stopping
pred_val_xgb  = xgb_model.predict(X_val_m_32)
pred_test_xgb = xgb_model.predict(X_test_m_32)

In [45]:
print("XGBoost (baseline):")
print("Val DA: ", directional_accuracy(y_val_vec, pred_val_xgb))

print("Test DA: ", directional_accuracy(y_test_vec, pred_test_xgb))

XGBoost (baseline):
Val DA:  0.4877632327831531
Test DA:  0.5093203556567544


Finetune on xgboost 

In [48]:
param_grid = {
    "max_depth":        [3, 4, 5],
    "learning_rate":    [0.03, 0.05, 0.1],
    "n_estimators":     [200, 400, 800],
    "subsample":        [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9],
}

results = []

for max_depth, lr, n_estimators, subsample, colsample in product(
    param_grid["max_depth"],
    param_grid["learning_rate"],
    param_grid["n_estimators"],
    param_grid["subsample"],
    param_grid["colsample_bytree"],
):
    params = {
        "max_depth": max_depth,
        "learning_rate": lr,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample,
    }

    model = xgb.XGBRegressor(
        **params,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    model.fit(X_train_m_32, y_train_vec, verbose=False)

    val_pred = model.predict(X_val_m_32)
    val_da   = directional_accuracy(y_val_vec, val_pred)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample,
        "val_DA": val_da,
    })

results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values("val_DA", ascending=False)
results_df_sorted.head()

,max_depth,learning_rate,n_estimators,subsample,colsample_bytree,val_DA
74,5,0.03,200,0.9,0.7,0.503996
48,4,0.05,200,0.7,0.7,0.503547
54,4,0.05,400,0.9,0.7,0.502601
44,4,0.03,800,0.7,0.7,0.502537
14,3,0.05,200,0.9,0.7,0.502337


In [71]:
best = results_df_sorted.iloc[0]
best_params = {
    "max_depth":        int(best["max_depth"]),
    "learning_rate":    float(best["learning_rate"]),
    "n_estimators":     int(best["n_estimators"]),
    "subsample":        float(best["subsample"]),
    "colsample_bytree": float(best["colsample_bytree"]),
}
print("Best params:", best_params)

best_xgb = xgb.XGBRegressor(
    **best_params,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

best_xgb.fit(X_train_m_32, y_train_vec, verbose=False)

pred_val_best  = best_xgb.predict(X_val_m_32)   # optional, just to inspect
pred_test_best = best_xgb.predict(X_test_m_32)

print("Val DA: ", directional_accuracy(y_val_vec,  pred_val_best))
print("Test DA:", directional_accuracy(y_test_vec, pred_test_best))

Best params: {'max_depth': 5, 'learning_rate': 0.03, 'n_estimators': 200, 'subsample': 0.9, 'colsample_bytree': 0.7}
Val DA:  0.5039960880822786
Test DA: 0.5185164399047519


Feature ablation

In [72]:
# inspect your columns once
feature_cols = list(X_train_m_32.columns)
feature_cols

['Close',
 's4_dow_mean',
 's7_roll63_mean_return',
 'ema_ema_50',
 'ema_ema_20',
 'ema_ema_10',
 'b0',
 'ma_1m',
 'ma_3d',
 'ma_14d',
 'ma_30d',
 'ma_50d',
 'ma_2w',
 'ma_2d',
 'ma_1w',
 's3_month_of_year_mean']

In [73]:
# base
base_cols = ["Close"]

# MA / EMA group (plus the long rolling mean s7)
ma_ema_cols = [
    c for c in feature_cols
    if c.startswith("ma_") or c.startswith("ema_") or c == "s7_roll63_mean_return"
]

# DOW + seasonality group
dow_season_cols = []
if "s4_dow_mean" in feature_cols:
    dow_season_cols.append("s4_dow_mean")
if "s3_month_of_year_mean" in feature_cols:
    dow_season_cols.append("s3_month_of_year_mean")

# simple lag return
b0_cols = ["b0"] if "b0" in feature_cols else []

print("MA/EMA group:", ma_ema_cols)
print("DOW/Seasonality group:", dow_season_cols)
print("Lag group:", b0_cols)

MA/EMA group: ['s7_roll63_mean_return', 'ema_ema_50', 'ema_ema_20', 'ema_ema_10', 'ma_1m', 'ma_3d', 'ma_14d', 'ma_30d', 'ma_50d', 'ma_2w', 'ma_2d', 'ma_1w']
DOW/Seasonality group: ['s4_dow_mean', 's3_month_of_year_mean']
Lag group: ['b0']


In [74]:
def run_xgb_da(feature_list, label):
    """
    Train XGBoost on a subset of features and report Val/Test DA.
    feature_list: list of column names to use.
    label: string name for printing / logging.
    """
    Xtr = X_train_m_32[feature_list]
    Xva = X_val_m_32[feature_list]
    Xte = X_test_m_32[feature_list]

    model = xgb.XGBRegressor(
        **best_params,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    model.fit(Xtr, y_train_vec, verbose=False)

    val_pred  = model.predict(Xva)
    test_pred = model.predict(Xte)

    val_da  = directional_accuracy(y_val_vec,  val_pred)
    test_da = directional_accuracy(y_test_vec, test_pred)

    print(f"{label}:")
    print(f"  Val DA:  {val_da:.4f}")
    print(f"  Test DA: {test_da:.4f}")
    print()
    
    return val_da, test_da

In [75]:
full_cols = feature_cols  # all columns in X_train_m_32
run_xgb_da(full_cols, "FULL MODEL")

FULL MODEL:
  Val DA:  0.5040
  Test DA: 0.5185



(np.float64(0.5039960880822786), np.float64(0.5185164399047519))

In [76]:
ma_ema_only_cols = base_cols + ma_ema_cols
run_xgb_da(ma_ema_only_cols, "ONLY MA/EMA + Close")

ONLY MA/EMA + Close:
  Val DA:  0.4901
  Test DA: 0.5187



(np.float64(0.49011198666100186), np.float64(0.5186607550891146))

In [78]:
dow_season_only_cols = base_cols + dow_season_cols
run_xgb_da(dow_season_only_cols, "ONLY DOW + Seasonality + Close")

ONLY DOW + Seasonality + Close:
  Val DA:  0.4942
  Test DA: 0.5136



(np.float64(0.49424034245312515), np.float64(0.5136177411466644))

In [79]:
no_ma_ema_cols = [c for c in feature_cols if c not in ma_ema_cols]
run_xgb_da(no_ma_ema_cols, "FULL - MA/EMA")

FULL - MA/EMA:
  Val DA:  0.4962
  Test DA: 0.5163



(np.float64(0.496172252639342), np.float64(0.5163196420983428))

In [80]:
no_dow_season_cols = [c for c in feature_cols if c not in dow_season_cols]
run_xgb_da(no_dow_season_cols, "FULL - DOW/Seasonality")

FULL - DOW/Seasonality:
  Val DA:  0.4925
  Test DA: 0.5185



(np.float64(0.492500821663046), np.float64(0.5184843698637825))

In [81]:
feature_names = X_train_m_32.columns

In [83]:
# assumes best_xgb is the trained model
booster = best_xgb.get_booster()
importance_dict = booster.get_score(importance_type="gain")

fi_gain = pd.DataFrame({
    "feature": list(importance_dict.keys()),
    "gain": list(importance_dict.values())
}).sort_values("gain", ascending=False)

fi_gain

,feature,gain
7,ma_1m,0.201014
1,s4_dow_mean,0.193971
13,ma_2d,0.160944
9,ma_14d,0.124181
14,ma_1w,0.115281
6,b0,0.111594
12,ma_2w,0.093945
10,ma_30d,0.093069
4,ema_ema_20,0.091173
3,ema_ema_50,0.086391


In [84]:
ranked_features = [
    "ma_1m",
    "s4_dow_mean",
    "ma_2d",
    "ma_14d",
    "ma_1w",
    "b0",
    "ma_2w",
    "ma_30d",
    "ema_ema_20",
    "ema_ema_50",
    "ma_3d",
    "ema_ema_10",
    "s7_roll63_mean_return",
    "ma_50d",
    "s3_month_of_year_mean",
    "Close",
]

In [85]:
for N in [3, 5, 8, 10, 12, 15]:
    keep = ranked_features[:N]
    run_xgb_da(keep, f"TOP {N} FEATURES")

TOP 3 FEATURES:
  Val DA:  0.4934
  Test DA: 0.5177

TOP 5 FEATURES:
  Val DA:  0.5001
  Test DA: 0.5148

TOP 8 FEATURES:
  Val DA:  0.5002
  Test DA: 0.5132

TOP 10 FEATURES:
  Val DA:  0.4989
  Test DA: 0.5160

TOP 12 FEATURES:
  Val DA:  0.5015
  Test DA: 0.5124

TOP 15 FEATURES:
  Val DA:  0.5016
  Test DA: 0.5168



In [86]:
# ========================================
# Top-K feature model (K chosen by Val DA)
# ========================================

# 1. Choose K based on your ablation results
K = 15  # Top 15 had the highest Val DA ≈ 0.5016

# ranked_features is already sorted by importance (highest → lowest)
topK = ranked_features[:K]
print(f"Top {K} features:", topK)

# -----------------------------
# (A) Train on TRAIN only → Val DA (no leakage)
# -----------------------------
xgb_topK_train = xgb.XGBRegressor(
    **best_params,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

xgb_topK_train.fit(X_train_m_32[topK], y_train_vec, verbose=False)

pred_val_topK = xgb_topK_train.predict(X_val_m_32[topK])
val_DA_topK   = directional_accuracy(y_val_vec, pred_val_topK)

print(f"Top-{K} Features (Train-only fit) — Val DA:", val_DA_topK)

# -----------------------------
# (B) Train on TRAIN+VAL → Test DA
# -----------------------------
X_trainval_topK = pd.concat(
    [X_train_m_32[topK], X_val_m_32[topK]],
    axis=0
)
y_trainval_vec = np.concatenate([y_train_vec, y_val_vec])

xgb_topK_trainval = xgb.XGBRegressor(
    **best_params,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

xgb_topK_trainval.fit(X_trainval_topK, y_trainval_vec, verbose=False)

pred_test_topK = xgb_topK_trainval.predict(X_test_m_32[topK])
test_DA_topK   = directional_accuracy(y_test_vec, pred_test_topK)

print(f"Top-{K} Features (Train+Val fit) — Test DA:", test_DA_topK)

Top 15 features: ['ma_1m', 's4_dow_mean', 'ma_2d', 'ma_14d', 'ma_1w', 'b0', 'ma_2w', 'ma_30d', 'ema_ema_20', 'ema_ema_50', 'ma_3d', 'ema_ema_10', 's7_roll63_mean_return', 'ma_50d', 's3_month_of_year_mean']
Top-15 Features (Train-only fit) — Val DA: 0.5016152693050735
Top-15 Features (Train+Val fit) — Test DA: 0.5221483720445453


In [87]:
# Directional accuracy for each model

# Linear Regression
val_DA_lr   = directional_accuracy(y_val_vec,  pred_val_lr)
test_DA_lr  = directional_accuracy(y_test_vec, pred_test_lr)

# XGBoost (tuned) with baseline features
val_DA_xgb  = directional_accuracy(y_val_vec,  pred_val_best)
test_DA_xgb = directional_accuracy(y_test_vec, pred_test_best)

# XGBoost (tuned) with Top-K signal features
val_DA_topK  = directional_accuracy(y_val_vec,  pred_val_topK)   # train-only → val
test_DA_topK = directional_accuracy(y_test_vec, pred_test_topK) # train+val → test

# Summary table
results_models = pd.DataFrame([
    {
        "model": "Linear Regression - baseline features only",
        "val_DA": val_DA_lr,
        "test_DA": test_DA_lr,
    },
    {
        "model": "XGBoost (tuned) - baseline features only",
        "val_DA": val_DA_xgb,
        "test_DA": test_DA_xgb,
    },
    {
        "model": f"XGBoost (tuned) - top {K} signal features",
        "val_DA": val_DA_topK,
        "test_DA": test_DA_topK,
    },
])

results_models

,model,val_DA,test_DA
0,Linear Regression - baseline features only,0.486689,0.495691
1,XGBoost (tuned) - baseline features only,0.503996,0.518516
2,XGBoost (tuned) - top 15 signal features,0.501615,0.522148
